# Proyecto 2 - Predicción de edad y género a partir de imágenes de caras

En este proyecto tendrás que desarrollar redes convolucionales capaces de predecir la edad y el género de una persona a partir de una fotografía de su rostro.

Deberás desarrollar tres modelos distintos:
 - Un modelo de regresión que haga predicciones de edad
 - Un modelo de clasificación que haga predicciones de género
 - Un modelo combinado que haga ambas predicciones a la vez

Tienes total libertad a la hora de diseñar la arquitectura de los modelos. Puedes utilizar técnicas de preprocesado. Se recomienda, para cada problema, diseñar un modelo desde cero y compararlo con un modelo adaptado a partir de una red preentrenada en otro problema mediante *transfer learning*.

### Competiciones

Podéis utilizar vuestros modelos para competir en las siguientes competiciones creadas en Kaggle para esta asignatura:
  - Predicción de edad:
    - Se mide con el error absoluto medio. Un valor razonable debe ser inferior a 15.
    - Competición para [modelos originales](https://www.kaggle.com/competitions/gced-aa-2-26-27-prediccion-edad).
    - Competición para [modelos preentrenados](https://www.kaggle.com/competitions/gced-aa-2-26-27-preentrenados-prediccion-edad).
    - Resultados de [otros años](https://www.kaggle.com/competitions/gced-aa2-2021-2022-edad/leaderboard) para comparar.
  - Predicción de género:
    - Se mide con la tasa de error de las predicciones. Un valor razonable debe ser inferior al 11%.
    - Competición para [modelos originales](https://www.kaggle.com/competitions/gced-aa-2-26-27-prediccion-sexo).
    - Competición para [modelos preentrenados](https://www.kaggle.com/competitions/gced-aa-2-26-27-preentrenados-prediccion-sexo).
    - Resultados de [otros años](https://www.kaggle.com/competitions/gced-aa2-2021-2022-sexo/leaderboard) para comparar.
  - Predicción de conjunta:
    - Se mide con la fórmula conjunta descrita en la página de la competición. Un valor razonable debe ser inferior a 26.
    - Competición para [modelos originales](https://www.kaggle.com/competitions/gced-aa-2-prediccion-conjunta).
    - Competición para [modelos preentrenados](https://www.kaggle.com/competitions/gced-aa-2-26-27-preentrenados-prediccion-conjunta).
    - Resultados de [otros años](https://www.kaggle.com/competitions/gced-aa2-2021-2022-edad-sexo/leaderboard) para comparar.

**Para cada competición, el mejor modelo podrá obtener una bonificación de 1 punto en la nota final de la asignatura**

# Código de apoyo para la carga del conjunto de datos

In [ ]:
!pip install gdown

# Descarga de los datos de entrenamiento
!gdown https://drive.google.com/u/0/uc?id=14Y-x-EhPKPUJ8s7LjHeFiS-gHcSRqQRN&export=download
# Descarga de los datos de test
!gdown https://drive.google.com/u/0/uc?id=1EZgAjfWcy_j9RqtUL7p9sPx9zpBAid2L&export=download

# Descompresión de los archivadores
!tar zxf dataset_proyecto.tar.gz
!tar zxf dataset_test_publico.tar.gz

# Eliminación de los archivadores para liberar espacio
!rm dataset_proyecto.tar.gz
!rm dataset_test_publico.tar.gz

# Comprobación
from os import listdir
import os.path

PATH = 'dataset_train_proyecto/'
PATH_TEST = 'dataset_test_publico/'

assert os.path.exists(PATH), 'Verifica los comandos anteriores'
assert os.path.exists(PATH_TEST), 'Verifica los comandos anteriores'

## Acceso a los metadatos

In [ ]:
import json

with open(PATH + 'metadata.json') as fIn:
    metadata = json.loads(fIn.read())
with open(PATH_TEST + 'metadata.json') as fIn:
    metadata_test = json.loads(fIn.read())

ejemplo_clave_metadata = next(iter(metadata.keys()))
ejemplo_metadata = metadata[ejemplo_clave_metadata]
print(ejemplo_clave_metadata, end=' ----> ')
print(ejemplo_metadata)

## Lectura de las imágenes

In [ ]:
import numpy as np
import torch
from torch.utils.data import Dataset, DataLoader
from torchvision.io import read_file, decode_jpeg, ImageReadMode
import torchvision.transforms.v2.functional as F
from matplotlib import pyplot

# Tamaño al que se procesarán las imágenes
img_height = 80
img_width = 80


def decode_img(img_bytes):
    '''Transforma los bytes de un JPEG a un tensor float32 (3, img_height, img_width)
    con valores en [0, 1].
    '''
    img = decode_jpeg(img_bytes, mode=ImageReadMode.RGB)       # uint8, (3, H, W)
    img = F.resize(img, [img_height, img_width], antialias=True)  # sigue en uint8
    return img.float() / 255.0                                  # float [0, 1]

GENDER_MAP = {'M': 0, 'F': 1}   # ajusta a los valores reales de tu metadata
def process_path(line):
    '''Transforma cadenas de la forma 'ruta|edad|genero' a tuplas (imagen, etiqueta) donde la etiqueta es una tupla (edad, género).
    '''
    path, age, gender = line.split('|')
    img = decode_img(read_file(path))
    age = torch.tensor(int(age), dtype=torch.int32)
    gender = torch.tensor(GENDER_MAP[gender], dtype=torch.int64)
    return img, (age, gender)


def formatea_metadata(m, path):
    '''Transforma el diccionario de metadatos m a una lista de cadenas de la forma
    'ruta a la imagen|edad|genero'
    '''
    return ['{0}{1}|{2}|{3}'.format(path, x, m[x]['age'], m[x]['gender']) for x in m.keys()]


class AgeAndGenderDataset(Dataset):
    '''Dataset de (imagen, edad) construido a partir de líneas 'ruta|edad|genero'.
    Las imágenes se cargan de disco bajo demanda en __getitem__.
    '''
    def __init__(self, lines):
        self.lines = list(lines)

    def __len__(self):
        return len(self.lines)

    def __getitem__(self, idx):
        return process_path(self.lines[idx])



DATASET_SIZE = len(metadata)
train_size = int(0.75 * DATASET_SIZE)

lineas_metadata = formatea_metadata(metadata, PATH)
np.random.shuffle(lineas_metadata)

train_ds = AgeAndGenderDataset(lineas_metadata[:train_size])
val_ds = AgeAndGenderDataset(lineas_metadata[train_size:])
test_ds = AgeAndGenderDataset(formatea_metadata(metadata_test, PATH_TEST))

# (Opcional) DataLoaders para entrenar
# train_dl = DataLoader(train_ds, batch_size=32, shuffle=True, num_workers=4)
# val_dl   = DataLoader(val_ds,   batch_size=32)
# test_dl  = DataLoader(test_ds,  batch_size=32)


# Comprobación
for i in range(5):
    image, label = train_ds[i]
    print(image.dtype)
    print("Image shape: ", tuple(image.shape))   # (3, 80, 80) -> canales primero
    print("Label: ", label)
    pyplot.imshow(image.permute(1, 2, 0).numpy())  # a (H, W, 3) para matplotlib
    pyplot.show()